In [1]:
from pathlib import Path
import duckdb
import pandas as pd

print("DuckDB:", duckdb.__version__)
print("pandas:", pd.__version__)

DATA = Path("data")
for f in ["articles.csv", "customers.csv", "transactions_train.csv"]:
    p = DATA / f
    size_mb = p.stat().st_size / 1024**2 if p.exists() else 0
    print(f"{f:<25} exists={p.exists()}  size={size_mb:,.1f} MB")

DuckDB: 1.5.6
pandas: 3.0.0
articles.csv              exists=True  size=34.5 MB
customers.csv             exists=True  size=197.5 MB
transactions_train.csv    exists=True  size=3,326.4 MB


In [4]:
import time

con = duckdb.connect()
con.execute("SET memory_limit = '8GB'")

jobs = {
    "articles": {"article_id": "VARCHAR", "product_code": "VARCHAR"},
    "customers": None,
    "transactions_train": {"article_id": "VARCHAR", "t_dat": "DATE"},
}

for name, types in jobs.items():
    start = time.time()
    src = (DATA / f"{name}.csv").as_posix()
    dst = (DATA / f"{name}.parquet").as_posix()
    types_sql = f", types={types}" if types else ""
    con.execute(f"""
        COPY (SELECT * FROM read_csv('{src}', header=true{types_sql}))
        TO '{dst}' (FORMAT parquet, COMPRESSION zstd)
    """)
    rows = con.execute(f"SELECT COUNT(*) FROM '{dst}'").fetchone()[0]
    size_mb = Path(dst).stat().st_size / 1024**2
    print(f"{name:<20} rows={rows:>12,}  parquet={size_mb:,.1f} MB  time={time.time()-start:.0f}s")

articles             rows=     105,542  parquet=3.3 MB  time=0s
customers            rows=   1,371,980  parquet=87.6 MB  time=1s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

transactions_train   rows=  31,788,324  parquet=436.3 MB  time=8s


In [5]:
for name in ["articles", "customers", "transactions_train"]:
    dst = (DATA / f"{name}.parquet").as_posix()
    print(f"\n=== {name} ===")
    print(con.execute(f"DESCRIBE SELECT * FROM '{dst}'").df()[["column_name", "column_type"]].to_string(index=False))

print("\nSample article_id:", con.execute(f"SELECT article_id FROM '{(DATA/'transactions_train.parquet').as_posix()}' LIMIT 3").fetchall())


=== articles ===
                 column_name column_type
                  article_id     VARCHAR
                product_code     VARCHAR
                   prod_name     VARCHAR
             product_type_no      BIGINT
           product_type_name     VARCHAR
          product_group_name     VARCHAR
     graphical_appearance_no      BIGINT
   graphical_appearance_name     VARCHAR
           colour_group_code     VARCHAR
           colour_group_name     VARCHAR
   perceived_colour_value_id      BIGINT
 perceived_colour_value_name     VARCHAR
  perceived_colour_master_id      BIGINT
perceived_colour_master_name     VARCHAR
               department_no      BIGINT
             department_name     VARCHAR
                  index_code     VARCHAR
                  index_name     VARCHAR
              index_group_no      BIGINT
            index_group_name     VARCHAR
                  section_no      BIGINT
                section_name     VARCHAR
            garment_group_no      BIGIN

In [6]:
tx = (DATA / "transactions_train.parquet").as_posix()

print(con.execute(f"DESCRIBE SELECT * FROM '{tx}'").df()[["column_name", "column_type"]].to_string(index=False))

print(con.execute(f"""
    SELECT MIN(t_dat) AS first_day,
           MAX(t_dat) AS last_day,
           COUNT(DISTINCT customer_id) AS customers,
           COUNT(DISTINCT article_id) AS articles
    FROM '{tx}'
""").df().to_string(index=False))

     column_name column_type
           t_dat        DATE
     customer_id     VARCHAR
      article_id     VARCHAR
           price      DOUBLE
sales_channel_id      BIGINT


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 first_day   last_day  customers  articles
2018-09-20 2020-09-22    1362281    104547
